# Part 3 — Final Evaluation

Compare all Part 3 models against each other and against Part 2 baselines.  
Produce the summary metrics table for the project report.

In [ ]:
import sys
sys.path.insert(0, "../..")

import pandas as pd
import numpy as np
import lightgbm as lgb
import joblib
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (
    accuracy_score, roc_auc_score, f1_score,
    confusion_matrix, RocCurveDisplay
)
import config

config.assert_data_exists()
sns.set_theme(style="whitegrid")

## 1. Load Test Data (2025)

In [ ]:
feat_path = config.DATA_PART3_PROCESSED / "flights_2023_2025_features.parquet"
df = pd.read_parquet(feat_path)

TARGET = "ARR_DEL15"
CAT_COLS = ["ORIGIN", "DEST", "OP_CARRIER"]
FEATURES = [c for c in df.columns if c not in [TARGET, "YEAR"]]

test_df = df[df["YEAR"] == 2025].copy()
X_test = test_df[FEATURES].copy()
y_test = test_df[TARGET]

for col in CAT_COLS:
    if col in X_test.columns:
        X_test[col] = X_test[col].astype("category")

print(f"Test set: {X_test.shape}  |  Delay rate: {y_test.mean():.3f}")

## 2. Load Models

In [ ]:
proc = config.DATA_PART3_PROCESSED

lgb_baseline = lgb.Booster(model_file=str(proc / "lgb_baseline.txt"))
lgb_tuned    = lgb.Booster(model_file=str(proc / "lgb_tuned.txt"))
xgb_model    = joblib.load(proc / "xgb_baseline.joblib")

## 3. Predict & Score

In [ ]:
def score(name, y_true, prob):
    pred = (prob >= 0.5).astype(int)
    return {
        "Model": name,
        "Accuracy": accuracy_score(y_true, pred),
        "ROC-AUC":  roc_auc_score(y_true, prob),
        "F1":       f1_score(y_true, pred),
    }

results = []

# LightGBM baseline
prob_lgb_base = lgb_baseline.predict(X_test)
results.append(score("LightGBM Baseline (Part 3)", y_test, prob_lgb_base))

# LightGBM tuned
prob_lgb_tuned = lgb_tuned.predict(X_test)
results.append(score("LightGBM Tuned (Part 3)", y_test, prob_lgb_tuned))

# XGBoost (need to re-encode cats as int codes for XGBoost)
X_test_xgb = X_test.copy()
for col in CAT_COLS:
    if col in X_test_xgb.columns:
        X_test_xgb[col] = X_test_xgb[col].cat.codes
prob_xgb = xgb_model.predict_proba(X_test_xgb)[:, 1]
results.append(score("XGBoost Baseline (Part 3)", y_test, prob_xgb))

# Part 2 reference numbers (hardcoded from 05_evaluation.ipynb)
results.append({"Model": "XGBoost (Part 2 reference)", "Accuracy": 0.799, "ROC-AUC": 0.848, "F1": None})
results.append({"Model": "PyTorch FFNN (Part 2 reference)", "Accuracy": 0.798, "ROC-AUC": 0.851, "F1": None})

summary = pd.DataFrame(results).set_index("Model").round(4)
summary

## 4. ROC Curves

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
for name, prob in [
    ("LightGBM Baseline", prob_lgb_base),
    ("LightGBM Tuned", prob_lgb_tuned),
    ("XGBoost", prob_xgb),
]:
    RocCurveDisplay.from_predictions(y_test, prob, name=name, ax=ax)
ax.set_title("ROC Curves — Part 3 (2025 test set)")
plt.tight_layout()
plt.show()

## 5. Confusion Matrix (Best Model)

In [ ]:
best_prob = prob_lgb_tuned  # update after running
best_pred = (best_prob >= 0.5).astype(int)

cm = confusion_matrix(y_test, best_pred, normalize="true")
sns.heatmap(cm, annot=True, fmt=".2%", cmap="Blues",
            xticklabels=["On-Time", "Delayed"],
            yticklabels=["On-Time", "Delayed"])
plt.title("Confusion Matrix — LightGBM Tuned (Part 3)")
plt.show()

## 6. Feature Importance (Tuned LightGBM)

In [ ]:
imp = pd.Series(
    lgb_tuned.feature_importance(importance_type="gain"),
    index=lgb_tuned.feature_name()
).sort_values(ascending=False)

imp.head(20).plot(kind="barh", figsize=(8, 8), title="Feature Importance — Tuned LightGBM (gain)")
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

## 7. Summary

TODO: fill in after running.

| | Accuracy | ROC-AUC | F1 |
|---|---|---|---|
| LightGBM Baseline (Part 3) | | | |
| LightGBM Tuned (Part 3)    | | | |
| XGBoost Baseline (Part 3)  | | | |
| XGBoost (Part 2 ref)       | 0.799 | 0.848 | — |
| PyTorch FFNN (Part 2 ref)  | 0.798 | 0.851 | — |